# 🌻 Perjalanan Yooji dan Mina Menonton Piala Dunia - Solusi Gammafest

**ID Tim**: DSC26068  
**Nama Tim**: Izin Numpang Lewat Bang

## Gambaran Umum

Notebook ini berisi solusi untuk kompetisi Gammafest dengan tujuan memprediksi skor pertandingan sepak bola internasional (`team_goals` dan `opp_goals`) menggunakan dataset historis dari tahun 1872 hingga 2026.

## Metrik Evaluasi: AW-MAE (Augmented Weighted Mean Absolute Error)

Metrik ini mempertimbangkan:
- Ketepatan skor akhir (Exact Score Penalty: 0.30)
- Ketepatan hasil pertandingan (Outcome Penalty: 0.25)
- Ketepatan selisih gol (Goal Difference Penalty: 0.15)
- Outcome Multiplier (1.5x jika outcome salah)
- Non-Linear Scaling (pangkat 1.5)
- Tournament Weighting (bobot berbeda per turnamen)

In [ ]:
# Impor Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries berhasil diimpor!")

In [ ]:
# Muat Data
print("Memuat data...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample submission.csv')

print(f"Bentuk data train: {train.shape}")
print(f"Bentuk data test: {test.shape}")
print(f"Bentuk sample submission: {sample_sub.shape}")

In [ ]:
# Eksplorasi Data Training
print("\n=== Informasi Data Training ===")
print(train.info())
print("\n=== 5 Baris Pertama ===")
display(train.head())
print("\n=== Distribusi Target ===")
print(train['team_goals'].describe())
print(train['opp_goals'].describe())

In [ ]:
# Eksplorasi Data Test
print("\n=== Informasi Data Test ===")
print(test.info())
print("\n=== 5 Baris Pertama ===")
display(test.head())

# Cek kolom yang ada di train tapi tidak ada di test (fitur performa hilang)
train_cols = set(train.columns)
test_cols = set(test.columns)
print(f"\nKolom di train tapi tidak ada di test: {train_cols - test_cols}")

In [ ]:
# Rekayasa Fitur
# Fitur yang tersedia di train dan test
common_features = ['is_home', 'neutral', 'confederation_team', 'confederation_opp', 
                   'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
                   'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue']

def prepare_features(df, is_train=True):
    df = df.copy()
    
    # Isi nilai kosong dengan median
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    # Encode variabel kategorikal konfederasi
    le_conf_team = LabelEncoder()
    le_conf_opp = LabelEncoder()
    
    all_conf_team = list(df['confederation_team'].fillna('Unknown').unique())
    all_conf_opp = list(df['confederation_opp'].fillna('Unknown').unique())
    
    le_conf_team.fit(all_conf_team)
    le_conf_opp.fit(all_conf_opp)
    
    df['confederation_team_enc'] = le_conf_team.transform(df['confederation_team'].fillna('Unknown'))
    df['confederation_opp_enc'] = le_conf_opp.transform(df['confederation_opp'].fillna('Unknown'))
    
    # Encoding gender (M=1, F=0)
    df['gender_enc'] = (df['gender'] == 'M').astype(int)
    
    return df

train_processed = prepare_features(train, is_train=True)
test_processed = prepare_features(test, is_train=False)

print("Rekayasa fitur selesai!")

In [ ]:
# Pilih fitur untuk pemodelan
feature_cols = ['is_home', 'neutral', 'confederation_team_enc', 'confederation_opp_enc',
                'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
                'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue',
                'gender_enc']

X_train = train_processed[feature_cols]
y_train_goals = train_processed['team_goals']
y_train_opp = train_processed['opp_goals']

X_test = test_processed[feature_cols]

print(f"Bentuk fitur training: {X_train.shape}")
print(f"Bentuk fitur test: {X_test.shape}")

In [ ]:
# Latih Model
print("Melatih model...")

# Menggunakan Gradient Boosting Regressor
model_goals = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
model_opp = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)

model_goals.fit(X_train, y_train_goals)
print("Model untuk team_goals selesai dilatih!")

model_opp.fit(X_train, y_train_opp)
print("Model untuk opp_goals selesai dilatih!")

In [ ]:
# Buat Prediksi
pred_goals = model_goals.predict(X_test)
pred_opp = model_opp.predict(X_test)

# Bulatkan prediksi ke bilangan bulat
pred_goals = np.round(pred_goals).astype(int)
pred_opp = np.round(pred_opp).astype(int)

# Pastikan prediksi tidak negatif
pred_goals = np.maximum(pred_goals, 0)
pred_opp = np.maximum(pred_opp, 0)

print(f"Prediksi berhasil dibuat! Bentuk: {len(pred_goals)}")
print(f"Rentang team_goals: {pred_goals.min()} - {pred_goals.max()}")
print(f"Rentang opp_goals: {pred_opp.min()} - {pred_opp.max()}")

In [ ]:
# Buat File Submission
submission = pd.DataFrame({
    'Id': test_processed['Id'],
    'team_goals': pred_goals,
    'opp_goals': pred_opp
})

# Simpan submission
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission tersimpan! Bentuk: {submission.shape}")
print("\n10 Baris pertama:")
display(submission.head(10))

In [ ]:
# Analisis Importansi Fitur
print("\n=== Importansi Fitur ===")
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance (goals)': model_goals.feature_importances_,
    'Importance (opp)': model_opp.feature_importances_
}).sort_values('Importance (goals)', ascending=False)

display(importance_df)

## Kesimpulan

Solusi ini menggunakan Gradient Boosting Regressor dengan fitur-fitur yang tersedia di kedua dataset (train dan test). Model dilatih secara terpisah untuk memprediksi `team_goals` dan `opp_goals`.

### Fitur yang Digunakan:
- `is_home`, `neutral`: Informasi lokasi pertandingan
- `confederation_team_enc`, `confederation_opp_enc`: Encoding konfederasi tim
- `population_team`, `population_opp`: Populasi negara
- `gdp_per_capita_team`, `gdp_per_capita_opp`: GDP per kapita
- `altitude_venue`, `temperature_venue`: Kondisi venue
- `distance_travel_team`, `distance_travel_opp`: Jarak perjalanan
- `gender_enc`: Gender pertandingan
